In [0]:
from databricks.sdk import WorkspaceClient
import pandas as pd

w = WorkspaceClient()

# Collect users with entitlements
rows = []
for user in w.users.list(attributes="id,userName,displayName,entitlements,active"):
    entitlements = [e.value for e in (user.entitlements or [])]
    if entitlements:
        rows.append({
            "type": "User",
            "id": user.id,
            "display_name": user.display_name or "",
            "user_name": user.user_name or "",
            "external_id": "",
            "active": user.active if user.active is not None else True,
            "entitlements": ", ".join(entitlements)
        })

# Collect service principals with entitlements
for sp in w.service_principals.list(attributes="id,applicationId,displayName,entitlements,active,externalId"):
    entitlements = [e.value for e in (sp.entitlements or [])]
    if entitlements:
        rows.append({
            "type": "Service Principal",
            "id": sp.id,
            "display_name": sp.display_name or "",
            "user_name": sp.application_id or "",
            "external_id": sp.external_id or "",
            "active": sp.active if sp.active is not None else True,
            "entitlements": ", ".join(entitlements)
        })

df = pd.DataFrame(rows, columns=["type", "id", "display_name", "user_name", "external_id", "active", "entitlements"])
print(f"Found {len(df[df['type']=='User'])} users and {len(df[df['type']=='Service Principal'])} service principals with entitlements")
display(df)

In [0]:
from databricks.sdk import WorkspaceClient
import pandas as pd

w = WorkspaceClient()

# Collect users with entitlements
rows = []
for user in w.users.list(attributes="id,userName,displayName,entitlements,active"):
    entitlements = [e.value for e in (user.entitlements or [])]
    rows.append({
        "type": "User",
        "id": user.id,
        "display_name": user.display_name or "",
        "user_name": user.user_name or "",
        "external_id": "",
        "active": user.active if user.active is not None else True,
        "entitlements": ", ".join(entitlements)
    })

# Collect service principals with entitlements
for sp in w.service_principals.list(attributes="id,applicationId,displayName,entitlements,active,externalId"):
    entitlements = [e.value for e in (sp.entitlements or [])]
    rows.append({
        "type": "Service Principal",
        "id": sp.id,
        "display_name": sp.display_name or "",
        "user_name": sp.application_id or "",
        "external_id": sp.external_id or "",
        "active": sp.active if sp.active is not None else True,
        "entitlements": ", ".join(entitlements)
    })

df = pd.DataFrame(rows, columns=["type", "id", "display_name", "user_name", "external_id", "active", "entitlements"])
print(f"Found {len(df[df['type']=='User'])} users and {len(df[df['type']=='Service Principal'])} service principals with entitlements")
display(df)

In [0]:
from databricks.sdk import WorkspaceClient
import pandas as pd

w = WorkspaceClient()

rows = []

# --- Direct entitlements for users ---
for user in w.users.list(attributes="id,userName,displayName,entitlements,active"):
    direct = [e.value for e in (user.entitlements or [])]
    if direct:
        rows.append({
            "type": "User",
            "id": user.id,
            "display_name": user.display_name or "",
            "identifier": user.user_name or "",
            "external_id": "",
            "active": user.active if user.active is not None else True,
            "entitlements": ", ".join(direct),
            "source": "direct"
        })

# --- Direct entitlements for service principals ---
for sp in w.service_principals.list(attributes="id,applicationId,displayName,entitlements,active,externalId"):
    direct = [e.value for e in (sp.entitlements or [])]
    if direct:
        rows.append({
            "type": "Service Principal",
            "id": sp.id,
            "display_name": sp.display_name or "",
            "identifier": sp.application_id or "",
            "external_id": sp.external_id or "",
            "active": sp.active if sp.active is not None else True,
            "entitlements": ", ".join(direct),
            "source": "direct"
        })

# --- Group-inherited entitlements ---
# Build a map of group_id -> list of entitlements
group_entitlements = {}
for group in w.groups.list(attributes="id,displayName,entitlements,members"):
    entitlements = [e.value for e in (group.entitlements or [])]
    if entitlements:
        group_entitlements[group.id] = {
            "group_name": group.display_name,
            "entitlements": entitlements,
            "members": group.members or []
        }

# Build lookup of all users and SPs for resolving member references
user_lookup = {}
for user in w.users.list(attributes="id,userName,displayName,active"):
    user_lookup[user.id] = {
        "type": "User",
        "display_name": user.display_name or "",
        "identifier": user.user_name or "",
        "external_id": "",
        "active": user.active if user.active is not None else True
    }

sp_lookup = {}
for sp in w.service_principals.list(attributes="id,applicationId,displayName,active,externalId"):
    sp_lookup[sp.id] = {
        "type": "Service Principal",
        "display_name": sp.display_name or "",
        "identifier": sp.application_id or "",
        "external_id": sp.external_id or "",
        "active": sp.active if sp.active is not None else True
    }

# Resolve group members and attach inherited entitlements
for group_id, info in group_entitlements.items():
    for member in info["members"]:
        member_id = member.value
        lookup_entry = user_lookup.get(member_id) or sp_lookup.get(member_id)
        if lookup_entry:
            rows.append({
                "type": lookup_entry["type"],
                "id": member_id,
                "display_name": lookup_entry["display_name"],
                "identifier": lookup_entry["identifier"],
                "external_id": lookup_entry["external_id"],
                "active": lookup_entry["active"],
                "entitlements": ", ".join(info["entitlements"]),
                "source": f"group: {info['group_name']}"
            })

df_all = pd.DataFrame(rows, columns=["type", "id", "display_name", "identifier", "external_id", "active", "entitlements", "source"])
df_all = df_all.sort_values(["type", "display_name", "source"]).reset_index(drop=True)

print(f"Total rows: {len(df_all)} ({len(df_all[df_all['source']=='direct'])} direct, {len(df_all[df_all['source']!='direct'])} group-inherited)")
display(df_all)